In [1]:
import numpy as np
import pandas as pd
import urllib
import json

/cvmfs/cdms.opensciencegrid.org/releases/centos7/V04-06/lib/python3.7/site-packages/pandas/compat/_optional.py:138: UserWarning: Pandas requires version '2.7.0' or newer of 'numexpr' (version '2.6.9' currently installed).
  warnings.warn(msg, UserWarning)


## Import metadata from Google Sheets

In [2]:
sheet_id = "1YklUMHrcqeEHhJtmiXgvCnb2B_SaIt99sZ-T6u1Kszc"
sheet_name = urllib.parse.quote("Raw data v2")
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/gviz/tq?tqx=out:csv&sheet={sheet_name}"

df = pd.read_csv(url)

In [3]:
# remove empty columns

df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

In [4]:
df.head()

,Facility,nFridgeRun,DCRCversion,DCRCFirmwareVersion,Series,nDataType,SourcePos_Cs,SourcePos_Ba,Category,Filetype,AnalysisTag,HV,Shield,WorkingPoint,MCTemperature,Path
0,25,13,RevE_SN19,a58c184,25220121_160513,noSource,NaN,NaN,Raw,.mid.gz,TRnoise,0,open,43/43/25/25/18/14/39/36,NaN,/sdf/group/supercdms/data/CDMS/NEXUS/R13/Raw
1,25,13,RevE_SN19,a58c184,25220121_161311,noSource,NaN,NaN,Raw,.mid.gz,SCnoise,0,open,0/0/0/0/0/0,NaN,/sdf/group/supercdms/data/CDMS/NEXUS/R13/Raw
2,25,13,RevE_SN19,a58c184,25220121_161617,noSource,NaN,NaN,Raw,.mid.gz,NRnoise,0,open,500/500/500/500/500/500,NaN,/sdf/group/supercdms/data/CDMS/NEXUS/R13/Raw
3,25,13,RevE_SN19,a58c184,25220121_172740,noSource,NaN,NaN,Raw,.mid.gz,100V_rate,100,open,43/43/25/25/18/14/39/39,16.0,/sdf/group/supercdms/data/CDMS/NEXUS/R13/Raw
4,25,13,RevE_SN19,a58c184,25220122_164022,noSource,NaN,NaN,Raw,.mid.gz,0V_rate,0,open,43/43/25/25/18/14/39/39,NaN,/sdf/group/supercdms/data/CDMS/NEXUS/R13/Raw


In [5]:
df.columns

Index(['Facility', 'nFridgeRun', 'DCRCversion', 'DCRCFirmwareVersion',
       'Series', 'nDataType', 'SourcePos_Cs', 'SourcePos_Ba', 'Category',
       'Filetype', 'AnalysisTag', 'HV', 'Shield', 'WorkingPoint',
       'MCTemperature', 'Path'],
      dtype='object')

## Data and functions used for conversion

In [6]:
facilities = {25: "NEXUS"}

In [7]:
n_data_types = {
    "Test": -1,
    "Background": 0,
    "Co": 1,
    "Co LowR": 2,
    "Cf": 3,
    "Rand": 4,
    "Mon": 7,
    "Cs source": 8,
    "Ba": 9,
    "YBe": 12,
    "SbBe": 13,
    "Y Blank": 14,
    "Sb Blank": 15,
    "Laser": 16,
    "Beam": 17,
    "Beam + Laser": 18,
    "Fe55 source": 19,
    "Fe55": 19,
    "Co57": 20,
    "IV curve": 100,
    "dIdV": 101,
    "NM noise": 102,
    "SC noise": 103,
    "TS noise": 104,
}

## Iterate by row and register the data

* Iterate through all rows in the Pandas Dataframe
* Build a dictionary of processed metadata
* Call `ContinuousRawData` with the `record` dictionary as arguments

In [8]:
row = df.to_dict(orient="records")[0]

In [9]:
row

{'Facility': 25,
 'nFridgeRun': 13,
 'DCRCversion': 'RevE_SN19',
 'DCRCFirmwareVersion': 'a58c184',
 'Series': '25220121_160513',
 'nDataType': 'noSource',
 'SourcePos_Cs': nan,
 'SourcePos_Ba': nan,
 'Category': 'Raw',
 'Filetype': '.mid.gz',
 'AnalysisTag': 'TRnoise',
 'HV': 0,
 'Shield': 'open',
 'WorkingPoint': '43/43/25/25/18/14/39/36',
 'MCTemperature': nan,
 'Path': '/sdf/group/supercdms/data/CDMS/NEXUS/R13/Raw'}

In [10]:
record = dict(
    fileName="FIXME",
    filePath=row.pop("Path"),
    facility=facilities[row.pop("Facility")],
    nFridgeRun=row.pop("nFridgeRun"),
    nDataType=n_data_types.get(row.pop("nDataType"), np.nan),
    Series=row.pop("Series"),
    nIsJunk=0,
    nDumpNum="FIXME",
)
record["commentStart"] = [json.dumps(row)]

In [11]:
record

{'fileName': 'FIXME',
 'filePath': '/sdf/group/supercdms/data/CDMS/NEXUS/R13/Raw',
 'facility': 'NEXUS',
 'nFridgeRun': 13,
 'nDataType': nan,
 'Series': '25220121_160513',
 'nIsJunk': 0,
 'nDumpNum': 'FIXME',
 'commentStart': ['{"DCRCversion": "RevE_SN19", "DCRCFirmwareVersion": "a58c184", "SourcePos_Cs": NaN, "SourcePos_Ba": NaN, "Category": "Raw", "Filetype": ".mid.gz", "AnalysisTag": "TRnoise", "HV": 0, "Shield": "open", "WorkingPoint": "43/43/25/25/18/14/39/36", "MCTemperature": NaN}']}